Movie-Recommender-System

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies.head() #just like imdb tmdb is website and id here is of that tmdb

In [ ]:
credits.head()

In [ ]:
credits.head(1)['cast'].values

In [ ]:
credits.head(1)['crew'].values

In [ ]:
movies.shape

In [ ]:
credits.shape

In [ ]:
# We don't need seperate data as credits and movies we need to merge this we can do this on the basis of movie_id or title
movies.merge(credits, on='title').shape

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
# we will remove what all columns that we have to keep
# we will keep genres, id, keywords
movies['original_language'].value_counts() # we will not keep original language

In [ ]:
# genres, id, keywords , title, overview cast and crew
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [ ]:
movies.info()

In [ ]:
movies.head() #this is dataframe on which we are going to work

In [ ]:
# we have to make a new dataframe this data('movie') and we will be having three column [movie-id|title|tags]
# we have to merge keywords , genres and overview
movies.head()

In [ ]:
# for preprocessing we will need to bring all these in correct format 

In [ ]:
movies.isnull().sum()  # to find missing data

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.isnull().sum()

In [ ]:
movies.iloc[0].genres #basicalky it is a list of dictionary 

In [ ]:
# '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [ ]:
def convertINV(obj):
    L = []
    for i in obj:
        L.append(i['name'])
    return L

In [ ]:
convertINV('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

In [ ]:
# we can't directly fed list of dictionary to our convert object firstly we will have to convert it into 
import ast 
ast.literal_eval('[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]')

In [ ]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'].apply(convert)

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies.head()

In [ ]:
movies['keywords'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter +=1
        else:
            break
    return L

In [ ]:
movies['cast'].apply(convert3)

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies.head()

In [ ]:
movies['crew'][0]

In [ ]:
# we have to find the name of crew who is director 
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [ ]:
movies['crew'].apply(fetch_director)

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head()

In [ ]:
movies['overview'][0] # we will have to convert overview column from string to list

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split()) # for converting into string

In [ ]:
movies.head()

In [ ]:
# we will remove the spacint between each names of each to form tag 
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])

In [ ]:
movies.head()

In [ ]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head()

In [ ]:
new_df = movies[['movie_id', 'title', 'tags']] #removing others columns after getting reque

In [ ]:
new_df.head()

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head()

In [ ]:
# we will convert text to vector which is called text-vectorisation and 
new_df['tags'][0]

In [ ]:
new_df['tags'][1]

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
ps.stem('dancing')

In [ ]:
ps.stem('loving')

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, stop_words='english')

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

In [ ]:
# here we will calculate cosine distance rather than euclidean distance
# distance high implies lower cosine distance
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
similarity[1]

In [ ]:
new_df[new_df['title'] == 'Batman Begins'].index[0]

In [ ]:
sorted(similarity[0], reverse = True)

In [ ]:
list(enumerate(similarity[0]))

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df['title']==movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse= True , key=lambda x:x[1])[1:6]

    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)
 

In [ ]:
new_df.iloc[1214]

In [ ]:
recommend('Avatar')

In [ ]:
recommend('Batman Begins')